# Data Preprocessing 

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import date
from pathlib import Path

import warnings

warnings.filterwarnings("ignore")

from etna.transforms import MedianOutliersTransform
from etna.transforms import TimeSeriesImputerTransform
import pandas as pd

from forecasting_sticker_sales.transform import convert_to_ts_df, convert_to_df

## Constants

In [ ]:
PROJECT_ROOT = Path("__file__").resolve().parents[1]

DATA_DPATH = PROJECT_ROOT / "data"
assert DATA_DPATH.exists(), "Data folder is not found"

SRC_DATA_DPATH = DATA_DPATH / "src_data"
assert SRC_DATA_DPATH.exists(), "Source data folder is not found"

OUTPUT_DATA_DPATH = DATA_DPATH / "preprocessed_data"
OUTPUT_DATA_DPATH.mkdir(parents=True, exist_ok=True)

SPLIT_DATE = date(year=2015, month=1, day=1)

## Data Loading 

In [ ]:
src_fpath = SRC_DATA_DPATH / "train.csv"

df = pd.read_csv(src_fpath)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by="date")

df.shape

In [ ]:
df.head()

## Train/Test Split

In [ ]:
train_df = df[df["date"].dt.date < SPLIT_DATE]
test_df = df[df["date"].dt.date >= SPLIT_DATE]

train_df.shape, test_df.shape

In [ ]:
train_df["date"].max(), test_df["date"].min()

In [ ]:
train_df["segment"] = train_df["country"] + "_" + train_df["store"] + "_" + train_df["product"]
test_df["segment"] = test_df["country"] + "_" + test_df["store"] + "_" + test_df["product"]

## Preprocessing 

### Outliers, Missing Values

In [ ]:
grouped = train_df.groupby(["segment"], as_index=False)

mean_sold = grouped["num_sold"].mean()
mean_sold["country"] = mean_sold["segment"].apply(lambda x: x.split("_")[0])
mean_sold["store"] = mean_sold["segment"].apply(lambda x: x.split("_")[1])
mean_sold["product"] = mean_sold["segment"].apply(lambda x: x.split("_")[2])

empty_segments = mean_sold[mean_sold["num_sold"].isna()]
empty_segments["country"] = empty_segments["segment"].apply(lambda x: x.split("_")[0])
empty_segments["store"] = empty_segments["segment"].apply(lambda x: x.split("_")[1])
empty_segments["product"] = empty_segments["segment"].apply(lambda x: x.split("_")[2])

empty_segments["fill_value"] = 0

for row_idx, row in empty_segments.iterrows():
    agg_data = mean_sold[
        (mean_sold["store"] == row["store"]) & (mean_sold["product"] == row["product"])
    ]
    empty_segments.loc[row_idx, "fill_value"] = agg_data["num_sold"].mean()

empty_segments

In [ ]:
# fill empty series with the constant value of the same stores and products
for _row_idx, row in empty_segments.iterrows():
    mask = train_df["segment"] == row["segment"]
    train_df.loc[mask, "num_sold"] = row["fill_value"]

In [ ]:
ts_train_df = convert_to_ts_df(train_df)
print(f"Source data shape: {ts_train_df.df.shape}")

outliers_remover = MedianOutliersTransform(in_column="target", window_size=15)
ts_train_df.fit_transform([outliers_remover])
print("Number of series with outliers:", len(outliers_remover.outliers_timestamps))
outliers_num = sum([len(values) for values in outliers_remover.outliers_timestamps.values()])
print(f"Total number of outliers: {outliers_num}")

imputer = TimeSeriesImputerTransform(in_column="target", strategy="running_mean", window=30)
ts_train_df.fit_transform([imputer])
print(f"Imputer Data Shape: {ts_train_df.df.shape}")

preprocessed_train_df = convert_to_df(ts_train_df)
print(f"Missing Values: {preprocessed_train_df['num_sold'].isna().sum()}")
preprocessed_train_df = preprocessed_train_df.dropna(subset=["num_sold"])

preprocessed_train_df.shape

In [ ]:
# check for correct null filling - must be 1 days for every train_df
df_check = preprocessed_train_df.copy()
df_check["date_shifted"] = df_check["date"].shift()
df_check = df_check[~df_check["date_shifted"].isna()]
print((df_check["date"] - df_check["date_shifted"]).max())

## Data Caching

In [ ]:
train_fpath = OUTPUT_DATA_DPATH / "train.csv"
preprocessed_train_df.to_csv(train_fpath)

test_fpath = OUTPUT_DATA_DPATH / "test.csv"
test_df.to_csv(test_fpath)